# Text generation with `vllm` as a library

[`vllm`](https://github.com/vllm-project/vllm) can be used both as a library and
a server. In this notebook, we focus on the library. Compared to `transformes`,
a lot of the complexity has been abstracted away.

In this notebook, we can now (finally!) use the AWQ version of Qwen-8B.

In [1]:
from vllm import LLM, SamplingParams

This can take a while, as the model is loaded and optimized:

In [2]:
model_name = "Qwen/Qwen3-8B-AWQ"
llm = LLM(model=model_name, max_model_len=16384, trust_remote_code=True)

INFO 03-23 15:50:19 [utils.py:238] non-default args: {'trust_remote_code': True, 'max_model_len': 16384, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-8B-AWQ'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 03-23 15:50:20 [model.py:531] Resolved architecture: Qwen3ForCausalLM
INFO 03-23 15:50:20 [model.py:1554] Using max model len 16384
INFO 03-23 15:50:21 [awq_marlin.py:162] The model is convertible to awq_marlin during runtime. Using awq_marlin kernel.
INFO 03-23 15:50:21 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.


Parse safetensors files:   0%|          | 0/2 [00:00<?, ?it/s]

INFO 03-23 15:50:22 [vllm.py:747] Asynchronous scheduling is enabled.
(EngineCore_DP0 pid=1026452) INFO 03-23 15:50:23 [core.py:101] Initializing a V1 LLM engine (v0.17.1) with config: model='Qwen/Qwen3-8B-AWQ', speculative_config=None, tokenizer='Qwen/Qwen3-8B-AWQ', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=16384, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=awq_marlin, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, ot

(EngineCore_DP0 pid=1026452) <frozen importlib._bootstrap_external>:1329: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore_DP0 pid=1026452) <frozen importlib._bootstrap_external>:1329: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


(EngineCore_DP0 pid=1026452) INFO 03-23 15:50:28 [default_loader.py:293] Loading weights took 1.74 seconds
(EngineCore_DP0 pid=1026452) INFO 03-23 15:50:29 [gpu_model_runner.py:4364] Model loading took 5.71 GiB memory and 3.357270 seconds
(EngineCore_DP0 pid=1026452) INFO 03-23 15:50:32 [decorators.py:465] Directly load AOT compilation from path /home/cwinkler/.cache/vllm/torch_compile_cache/torch_aot_compile/4d44860c27c7f2e8dd11618289846b7c7ee1520a97b6575f9474c11356059668/rank_0_0/model
(EngineCore_DP0 pid=1026452) INFO 03-23 15:50:32 [backends.py:916] Using cache directory: /home/cwinkler/.cache/vllm/torch_compile_cache/e44751665b/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=1026452) INFO 03-23 15:50:32 [backends.py:976] Dynamo bytecode transform time: 3.01 s
(EngineCore_DP0 pid=1026452) INFO 03-23 15:50:34 [backends.py:266] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.310 s
(EngineCore_DP0 pid=1026452) INFO 03-23 15:50:34 [

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|█████████████████████████████████████████████████████████████| 51/51 [00:02<00:00, 21.76it/s]
Capturing CUDA graphs (decode, FULL): 100%|████████████████████████████████████████████████████████████████████████████████| 35/35 [00:01<00:00, 26.67it/s]


(EngineCore_DP0 pid=1026452) INFO 03-23 15:50:46 [gpu_model_runner.py:5386] Graph capturing finished in 4 secs, took 0.58 GiB
(EngineCore_DP0 pid=1026452) INFO 03-23 15:50:46 [core.py:282] init engine (profile, create kv cache, warmup model) took 17.46 seconds
INFO 03-23 15:50:48 [llm.py:388] Supported tasks: ['generate']


The `messages` are exactly the same:

In [3]:
messages = [
    {"role": "system", "content": "Du bist ein hilfreicher Assisten."},
    {"role": "user", "content": "Erkläre den Heise Verlag!"}
]

Sampling parameters are passed separately.

In [4]:
sampling_params = SamplingParams(
  max_tokens=1024,
  temperature=0.0,
)

Avoid the thinking phase:

In [5]:
output = llm.chat(messages=messages, sampling_params=sampling_params, 
                  chat_template_kwargs={"enable_thinking": False})

Rendering conversations:   0%|          | 0/1 [00:00<?, ?it/s]

INFO 03-23 15:50:49 [hf.py:318] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


Processed prompts:   0%|                                                         | 0/1 [00:00<?, ?it/s, est. s…

In [6]:
print(output[0].outputs[0].text)

Der **Heise Verlag** ist ein deutscher Verlag, der sich auf die Veröffentlichung von **Technik- und IT-Beiträgen** spezialisiert hat. Er ist besonders bekannt für seine **Computer-Magazine** und **Online-Plattformen**, die sich auf Themen wie **Software, Hardware, Internet, Sicherheit, Programmierung** und **Informatik** konzentrieren.

### Hauptprodukte und Plattformen des Heise Verlags:

1. **Heise Online**  
   - Die **Hauptwebsite** des Verlags, auf der viele IT- und Technik-Themen behandelt werden.
   - Enthält **Artikel, News, Tutorials, Reviews** und **Forum-Beiträge**.
   - Die Plattform ist **kostenlos** zugänglich, aber manche Inhalte erfordern eine **Registrierung**.

2. **Heise Magazin**  
   - Ein **monatlich erscheinendes Printmagazin**, das sich auf **IT- und Technik-Themen** konzentriert.
   - Es wird in **Deutschland** und **Schweiz** vertrieben.
   - Das Magazin ist **kostenpflichtig** und kann über den Heise Verlag oder im **Zeitschriftenhandel** erworben werden.

3.

In [7]:
!nvidia-smi

Mon Mar 23 15:50:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.48.01              Driver Version: 590.48.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        Off |   00000000:26:00.0 Off |                  Off |
| 30%   45C    P2            289W /  450W |   22798MiB /  24564MiB |     32%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----